In [90]:
from linearmodels import PanelOLS
import pandas as pd

df = pd.read_excel('data.xlsx', index_col='state_name', na_values=['..'])

df = df.drop(index=['Algeria', 'Egypt', 'Tunisia', 'Morocco'])
# average_gdp_per_capita = df.groupby('state_name')['gdp_per_capita'].mean()
# countries_to_keep = average_gdp_per_capita[average_gdp_per_capita <= df['gdp_per_capita'].quantile(0.75)].index
# df = df[df.index.isin(countries_to_keep)]

In [91]:
# посчитаем среднее ежегодное миграционное сальдо (номинальное и взвешенное) у всех стран в выборке за рассматриваемый период
average_migration_balance = df.groupby('state_name').agg(average_annual_migration_balance=('migration_balance', 'mean')).astype('int64').reset_index()
average_migration_balance = average_migration_balance.merge(df.groupby('state_name').agg(average_annual_migration_balance_percent=('m_balance_percent', 'mean')).reset_index())
average_migration_balance['average_annual_migration_balance_percent'] = average_migration_balance['average_annual_migration_balance_percent'].round(4)

# добавим колонку со средним значением населения в странах за рассматриваемый период для оценки масштабов миграции
average_migration_balance = average_migration_balance.merge(df.groupby('state_name').agg(average_population=('population', 'mean')).astype('int64').reset_index())

# отсортируем, вынесем названия стран в индекс
average_migration_balance = average_migration_balance.sort_values('average_annual_migration_balance_percent').set_index('state_name')

# поставим запятую в качестве разделителя тысяч для удобства чтения
average_migration_balance = average_migration_balance.style.format(
    {'average_annual_migration_balance': '{:,.0f}', 
     'average_annual_migration_balance_percent': '{:.4f}',
     'average_population': '{:,.0f}'})

average_migration_balance

,average_annual_migration_balance,average_annual_migration_balance_percent,average_population
state_name,,,
Central African Republic,"-53,804",-1.1195,"4,685,433"
Eritrea,"-34,798",-1.0438,"3,346,822"
Zimbabwe,"-94,180",-0.7275,"13,531,470"
Lesotho,"-9,135",-0.4525,"2,079,188"
Cabo Verde,"-2,285",-0.4425,"531,231"
Côte d'Ivoire,"-83,970",-0.4095,"22,118,185"
Namibia,"-7,653",-0.3545,"2,174,831"
Comoros,"-2,073",-0.2846,"730,950"
Guinea,"-27,829",-0.2820,"10,809,374"


In [93]:
# посчитаем сумму миграционного сальдо у каждой из стран за период 2010-2022 и сравним с приростом населения за этот период
migration_balance_sum = df[df['year'] >= 2010].groupby('state_name').agg(migration_balance_sum=('migration_balance', 'sum')).reset_index()
migration_balance_sum = migration_balance_sum.merge(df[df['year'] == 2010][['population']].astype('int64').reset_index()).rename(columns={'population': 'population_2010'})
migration_balance_sum = migration_balance_sum.merge(df[df['year'] == 2021]['population'].astype('int64').reset_index()).rename(columns={'population': 'population_2021'})
migration_balance_sum = migration_balance_sum.sort_values('migration_balance_sum').set_index('state_name')
migration_balance_sum['population_2010_2021_diff'] = df[df['year'] == 2021]['population'] - df[df['year'] == 2010]['population']

# запятые в качестве разделителя тысяч
migration_balance_sum = migration_balance_sum.style.format(
    {'migration_balance_sum': '{:,.0f}', 
     'population_2010': '{:,.0f}',
     'population_2021': '{:,.0f}',
     'population_2010_2021_diff': '{:,.0f}'})

migration_balance_sum

,migration_balance_sum,population_2010,population_2021,population_2010_2021_diff
state_name,,,,
Central African Republic,"-904,272","4,645,455","5,440,945","795,490"
Zimbabwe,"-807,931","12,872,340","15,994,805","3,122,465"
Côte d'Ivoire,"-737,206","21,145,455","27,502,146","6,356,691"
Kenya,"-680,297","41,391,667","52,893,939","11,502,272"
Mali,"-545,287","15,587,302","21,876,712","6,289,410"
Eritrea,"-428,025","3,141,509","3,621,160","479,651"
Nigeria,"-392,500","160,956,627","213,475,610","52,518,983"
Uganda,"-377,631","32,412,791","45,880,886","13,468,095"
Burundi,"-350,810","13,000,000","12,736,842","-263,158"


In [94]:
# построим [панельную] регрессионную модель
df = df.reset_index().set_index(['state_name', 'year'])

dependent_var = df['m_balance_percent']
independent_vars = df[['gdp_per_capita', 'fdi_per_capita', 'hdi_norm', 'tax_burden', 'government_spending', 'trade_freedom', 'gov_effectiveness', 'pstab', 'fatalities_per_10000']]

model = PanelOLS(dependent_var, independent_vars, entity_effects=False)
results = model.fit()

results.summary

Dep. Variable:,m_balance_percent,R-squared:,0.3340
Estimator:,PanelOLS,R-squared (Between):,0.5421
No. Observations:,777,R-squared (Within):,-0.0162
Date:,"Sat, Sep 14 2024",R-squared (Overall):,0.3340
Time:,20:31:19,Log-likelihood,-429.87
Cov. Estimator:,Unadjusted,,
,,F-statistic:,42.786
Entities:,41,P-value,0.0000
Avg Obs:,18.951,Distribution:,"F(9,768)"
Min Obs:,13.000,,
Max Obs:,20.000,F-statistic (robust):,42.786


In [55]:
# экспорт данных в txt-файл
summary_text = results.summary.as_text()
with open('regression_results.txt', 'w') as f:
    f.write(summary_text)